# 추론 자체 검증
품질 정답 정확도 검증이 아니라 기존 적용본 일치·배치 불변성·정답 미사용·판정보류 검사.

In [1]:
from pathlib import Path
import json,nbformat,pandas as pd,numpy as np
root=Path.cwd()
if not (root/'models').exists():root=root.parent
exec(compile(nbformat.read(root/'notebooks/154_rg3_gas_model_export.ipynb',as_version=4).cells[1].source,'rg3_inference_helpers','exec'))
bundle=load_bundle(root/'models/rg3_gas_final_fit.joblib')
query=pd.read_csv(root/'examples/rg3_gas_raw_input.csv')
history=pd.read_csv(root/'examples/rg3_gas_raw_history.csv')
expected=pd.read_csv(root/'examples/rg3_gas_expected_output.csv',parse_dates=['TimeStamp'])
features=prepare_features(query,history)
pred=predict_cascade(bundle,features)
check=pred.merge(expected,on=KEYS,suffixes=('_new','_expected'),validate='one_to_one')
assert len(check)==5 and np.array_equal(check.gas_prediction_new,check.gas_prediction_expected)
# No inference-time fit, even on peer reference objects or scalers.
classes=[RandomForestClassifier,ExtraTreesClassifier,Ridge,LogisticRegression,RobustScaler,SimpleImputer,KMeans,NearestNeighbors]
oldfits={c:c.fit for c in classes}
def forbidden_fit(*a,**kw):raise AssertionError('Inference attempted to fit a model')
try:
 for c in classes:c.fit=forbidden_fit
 again=predict_cascade(bundle,features)
finally:
 for c,m in oldfits.items():c.fit=m
assert np.array_equal(pred.gas_prediction,again.gas_prediction)
singles=pd.concat([predict_gas_features(bundle,features.iloc[[i]]) for i in range(len(features))],ignore_index=True)
assert np.array_equal(pred.gas_prediction,singles.gas_prediction)
with_labels=features.copy();with_labels['PassOrFail']=1;with_labels['Reason']='fake_label'
assert np.array_equal(pred.gas_prediction,predict_gas_features(bundle,with_labels).gas_prediction)
tiny=prepare_features(query.head(1))
assert predict_cascade(bundle,tiny).status.eq('판정보류').all()
other=features.copy();other['EQUIP_CD']='UNSUPPORTED'
assert predict_cascade(bundle,other).status.eq('판정보류').all()
# Interface-only routing test: temporary thresholds force gas-negative on fixture.
mock=dict(bundle);mock['cuts']={**bundle['cuts']}
for k in ['baseline','expert','thermal_score','pressure_42_7_101','pressure_11_23_37','pressure_13_29_53']:mock['cuts'][k]=np.inf
negative=predict_cascade(mock,features)
assert negative.predicted_type.eq('미성형 모델 미연결').all()
seen=[]
def mock_shortshot(q):seen.append(len(q));return np.zeros(len(q),int)
routed=predict_cascade(mock,features,mock_shortshot)
assert seen==[5] and routed.predicted_type.eq('정상 예측').all()
# An all-gas input must never invoke the stage2 callback.
seen.clear();predict_cascade(bundle,features,mock_shortshot);assert not seen
result=dict(artifact_sha_verified=True,fixture_rows=5,fixture_matches=True,no_inference_fit=True,batch_invariant=True,label_invariant=True,missing_history_abstains=True,unsupported_equipment_abstains=True,cascade_interface_routing=True,shortshot_model_connected=False,independent_quality_validation=False)
(root/'reports/tables/rg3_inference_portable_selfcheck.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "artifact_sha_verified": true,
  "fixture_rows": 5,
  "fixture_matches": true,
  "no_inference_fit": true,
  "batch_invariant": true,
  "label_invariant": true,
  "missing_history_abstains": true,
  "unsupported_equipment_abstains": true,
  "cascade_interface_routing": true,
  "shortshot_model_connected": false,
  "independent_quality_validation": false
}
